# Maze where turning is expensive

In [1]:
with open("inputs/16", "r") as fp:
    data = fp.read()

In [2]:
lines = data[:-1].split("\n")

In [3]:
l0 = len(lines)
l1 = len(lines[0])

In [4]:
def get_tiles_x(lines, c):
    for idx, line in enumerate(lines):
        for jdx, ci in enumerate(line):
            if ci == c:
                return (idx, jdx)

In [5]:
pt_end = get_tiles_x(lines, "E")
pt_start = get_tiles_x(lines, "S")


In [6]:
direction = ">"

## Approach

Keep several path in parallels.

Improve the path with the lowest score.

Should not go back on previous path (discourage because expensive to rotate)


In [7]:
actions = [("move", 1), ("turn_r", 1000), ("turn_l", 1000)]

dic_turn = {
    "turn_r": {">": "v", "v": "<", "<": "^", "^": ">"},
    "turn_l": {">": "^", "v": ">", "<": "v", "^": "<"},
}
dic_move = {">": (0, 1), "<": (0, -1), "^": (-1, 0), "v": (1, 0)}


def update_location(pt, direction, s0):
    """
    :param pt: current location (r, c)
    :param direction: v, ^, <, >
    :param s0: current score
    """
    lst = [] # New points + score
    for action, score in actions:
        if action == "move":
            dx, dy = dic_move[direction]
            x1, y1 = pt[0] + dx, pt[1] + dy
            if lines[x1][y1] != "#":
                lst.append(((x1, y1), direction, s0+score))
        else:
            dir_new = dic_turn[action][direction]
            lst.append((pt, dir_new, s0+score))

    return lst

In [8]:
def loop_explore_v2(pt0, pte, dir0=">", dir1=">"):
    """
    dir0 and dir1 help for part 2
    """
    infinite = 10000000000
    ref_score = {
        "^": [[infinite for _ in line] for line in lines],
        "v": [[infinite for _ in line] for line in lines],
        "<": [[infinite for _ in line] for line in lines],
        ">": [[infinite for _ in line] for line in lines]}

    #for c in "^v><":
    #    ref_score[c][pt0[0]][pt0[1]] = 0
    ref_score[dir0][pt0[0]][pt0[1]] = 0
    
    # Start condition
    lst = [(pt0, dir0, 0)]
    lst_end = []
    for steps in range(1000):
        #print(lst)
        if len(lst) == 0:
            break 
            
        lst_new = []
        for pti, direction, s0 in lst:
            lst_new.extend(update_location(pti, direction, s0))


        # Filter to remove points that are helpless
        lst = []
        for pt, d, sc in lst_new:
            if ref_score[d][pt[0]][pt[1]] > sc:
                # Keep only if score ok
                ref_score[d][pt[0]][pt[1]] = sc
                lst.append((pt, d, sc))

        # Remove duplicated points
        dic = {"v": {}, "^": {}, ">": {}, "<": {}}
        for pt, d, sc in lst:
            label = "{}_{}".format(pt[0], pt[1])
            if label not in dic[d]:
                dic[d][label] = [pt, sc]
            
            else:
                # Keep point with lowest score
                _, sc0 = dic[d][label]
                dic[d][label] = [pt, min(sc, sc0)]
        
        lst = []
        for d, vals in dic.items():
            for pt, sc in vals.values():
                lst.append((pt, d, sc))
        
        # Check end condition
        for pt, d, sc in lst:
            if (pt[0] == pte[0]) & (pt[1] == pte[1]) & (dir1 == d):
                lst_end.append(sc)
                

    return lst, lst_end, ref_score
        
            

# Part 1+2

Count how many items are on the best paths.

Easy:

Check E-> S and S-> E best score matrix

In [9]:
A0 = loop_explore_v2(pt_start, pt_end)[2]
print("OK")
A1 = loop_explore_v2(pt_end, pt_start, "<", )[2]
print("OK")



OK
OK


In [13]:
lines_copy = list(map(lambda x: [y for y in x], lines))

cnt = 0
for c0, c1 in [(">","<"), ("<",">"), ("^","v"), ("v", "^")]:
    A_tot = [[a + b for a, b in zip(A0_line, A1_line)] for A0_line, A1_line in zip(A0[c0], A1[c1])]

    v_min = min(list(map(min, A_tot)))
    print(v_min, "(possible P1 solution)")
    s0 = sum([sum([x == v_min for x in line]) for line in A_tot])
    cnt += s0
    for idx, line in enumerate(A_tot):
        for jdx, c in enumerate(line):
            if c == v_min:
                lines_copy[idx][jdx] = "o"
    
print("===", cnt, "===")
print("Maybe there are duplicates. Check next")

88468 (possible P1 solution)
88468 (possible P1 solution)
88468 (possible P1 solution)
88468 (possible P1 solution)
=== 744 ===
Maybe there are duplicates. Check next


In [14]:
print("\n".join(list(map(lambda x: "".join(x), lines_copy))))

#############################################################################################################################################
#.........#.............#.....#.....#.....#.........#...#.............#...#.............#.........#.....#...................#.....#.#ooooooo#
#.#######.#.###########.#.###.###.#.#.#.#.#.#.#.###.###.#.###########.#.###.#####.#######.#.###.#.###.#.#.###########.#.###.###.#.#.#o###o###
#.......................#.#.....#.#.#.#.#.#...#...#...#.#.#.......#...#.......................#.#...............#......ooooooooooooooooooo..#
#.#.#.#.#####.#####.#.###.#####.#.#.###.#.###.###.###.#.#.#.#.#.###.###.#.###.#.#.#.#########.#.#####.#######.#.#.#.###o#.###.#.#.###.#####.#
#...#.#.#...#.#...#...#...#...#...#...#.#.........#.....#.#.#.#.#...#...#.#...#.#.....#.....#.#.....#.....#.#.#.#.#....o#.............#.....#
#.###.###.#.###.#.#.#####.#.#.#######.#.#####.#.#######.#.#.#.###.###.###.#.###########.###.#######.#.###.#.#.#.#.#####o#.#######.#.###.#####
#.#.#.

In [15]:
print("Part 2 solution:", sum([sum([x == "o" for x in line]) for line in lines_copy]))

Part 2 solution: 616
